<a href="https://colab.research.google.com/github/nomadcodin/XAI_HybridNet/blob/main/Computational_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install timm -q

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import timm
import time
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
resnet = models.resnet50(weights=None)

resnet.fc = nn.Linear(
    resnet.fc.in_features,
    2
)

resnet.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/resnet50_chestxray.pth",
        map_location=device
    )
)

resnet = resnet.to(device)
resnet.eval()

print("ResNet-50 loaded successfully")

In [ ]:
vit = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False
)

vit.head = nn.Linear(
    vit.head.in_features,
    2
)

vit.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/vit_chestxray.pth",
        map_location=device
    )
)

vit = vit.to(device)
vit.eval()

print("ViT-B/16 loaded successfully")

In [ ]:
dummy = torch.randn(
    1,
    3,
    224,
    224
).to(device)

print(dummy.shape)

In [ ]:
# Warmup

for _ in range(20):
    with torch.no_grad():
        _ = resnet(dummy)

runs = 100

torch.cuda.synchronize()

start = time.time()

for _ in range(runs):
    with torch.no_grad():
        _ = resnet(dummy)

torch.cuda.synchronize()

end = time.time()

resnet_latency = ((end - start) / runs) * 1000

print(
    f"ResNet Latency: {resnet_latency:.2f} ms/image"
)

In [ ]:
for _ in range(20):
    with torch.no_grad():
        _ = vit(dummy)

runs = 100

torch.cuda.synchronize()

start = time.time()

for _ in range(runs):
    with torch.no_grad():
        _ = vit(dummy)

torch.cuda.synchronize()

end = time.time()

vit_latency = ((end - start) / runs) * 1000

print(
    f"ViT Latency: {vit_latency:.2f} ms/image"
)

In [ ]:
for _ in range(20):
    with torch.no_grad():

        r = resnet(dummy)
        v = vit(dummy)

        r = torch.softmax(r, dim=1)
        v = torch.softmax(v, dim=1)

        hybrid = 0.5*r + 0.5*v

runs = 100

torch.cuda.synchronize()

start = time.time()

for _ in range(runs):
    with torch.no_grad():

        r = resnet(dummy)
        v = vit(dummy)

        r = torch.softmax(r, dim=1)
        v = torch.softmax(v, dim=1)

        hybrid = 0.5*r + 0.5*v

torch.cuda.synchronize()

end = time.time()

hybrid_latency = ((end - start) / runs) * 1000

print(
    f"HybridNet Latency: {hybrid_latency:.2f} ms/image"
)

In [ ]:
torch.cuda.empty_cache()

torch.cuda.reset_peak_memory_stats()

with torch.no_grad():
    _ = resnet(dummy)

torch.cuda.synchronize()

resnet_memory = (
    torch.cuda.max_memory_allocated()
    / (1024**2)
)

print(
    f"ResNet Memory: {resnet_memory:.2f} MB"
)

In [ ]:
torch.cuda.empty_cache()

torch.cuda.reset_peak_memory_stats()

with torch.no_grad():
    _ = vit(dummy)

torch.cuda.synchronize()

vit_memory = (
    torch.cuda.max_memory_allocated()
    / (1024**2)
)

print(
    f"ViT Memory: {vit_memory:.2f} MB"
)

In [ ]:
torch.cuda.empty_cache()

torch.cuda.reset_peak_memory_stats()

with torch.no_grad():

    r = resnet(dummy)
    v = vit(dummy)

    r = torch.softmax(r, dim=1)
    v = torch.softmax(v, dim=1)

    hybrid = 0.5*r + 0.5*v

torch.cuda.synchronize()

hybrid_memory = (
    torch.cuda.max_memory_allocated()
    / (1024**2)
)

print(
    f"Hybrid Memory: {hybrid_memory:.2f} MB"
)

In [ ]:
results = pd.DataFrame({

    "Model":[
        "ResNet-50",
        "ViT-B/16",
        "HybridNet"
    ],

    "Latency (ms/image)":[
        round(resnet_latency,2),
        round(vit_latency,2),
        round(hybrid_latency,2)
    ],

    "Peak GPU Memory (MB)":[
        round(resnet_memory,2),
        round(vit_memory,2),
        round(hybrid_memory,2)
    ]
})

results